# B2B Sales Intelligence Agent — demo end-to-end

Este notebook ejecuta el pipeline completo (plan → execute → reflect → build → hooks → email → score) con los **stubs deterministas**: no necesita `ANTHROPIC_API_KEY` ni `TAVILY_API_KEY`. Con llaves configuradas en `.env`, las mismas celdas usan Claude y Tavily y el campo `mode` cambia de `fallback` a `llm`.

In [1]:
import sys, json, asyncio
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.environ.setdefault('USE_REAL_HN', 'false')
from src.config import get_settings
from src.observability import configure_logging
configure_logging('WARNING')
s = get_settings()
print('model:', s.anthropic_model, '| llm_enabled:', s.llm_enabled, '| tavily:', bool(s.tavily_api_key))

model: claude-sonnet-4-5-20250929 | llm_enabled: False | tavily: False


## 1. Construir el grafo y investigar una empresa

In [2]:
from src.agents.orchestrator import build_components, build_graph, research_company
components = build_components(s)
graph = build_graph(components)
result = asyncio.run(
    research_company(graph, company_name='Rescale', website='rescale.com')
)
print('mode:', result.mode, '| iterations:', result.iterations, '| tool_calls:', result.tool_calls, '| latency_ms:', result.latency_ms, '| cost_usd:', result.cost_usd)

mode: fallback | iterations: 1 | tool_calls: 7 | latency_ms: 8 | cost_usd: 0.0


## 2. Perfil estructurado (`CompanyProfile`)

In [3]:
print(result.profile.model_dump_json(indent=2))

{
  "name": "Rescale",
  "one_liner": "Modern B2B software platform.",
  "industry": null,
  "website": "rescale.com",
  "employees_est": null,
  "funding_total_usd": 40000000.0,
  "last_funding_round": "Series B",
  "location": null,
  "key_people": [
    {
      "name": "John Doe",
      "role": "CEO",
      "linkedin_url": null
    },
    {
      "name": "Jane Smith",
      "role": "CTO",
      "linkedin_url": null
    }
  ],
  "recent_signals": [
    {
      "kind": "funding",
      "description": "Rescale Recent Funding Round raises Series B",
      "source_url": "https://techcrunch.com/example-funding",
      "occurred_at": null
    },
    {
      "kind": "news",
      "description": "Show HN: Rescale launches new feature",
      "source_url": "https://news.ycombinator.com/item?id=12345",
      "occurred_at": null
    },
    {
      "kind": "news",
      "description": "Ask HN: experiences with Rescale?",
      "source_url": "https://news.ycombinator.com/item?id=12346",
      "oc

## 3. Ganchos de personalización y email

In [4]:
print(result.hooks.model_dump_json(indent=2))
print()
print('Subject:', result.email.subject)
print(result.email.body())

{
  "pain_point": "post-raise growth targets outpace what manual operations can deliver",
  "recent_signal": "funding: Rescale Recent Funding Round raises Series B",
  "angle": "Because Rescale just showed funding, automating the workflows behind that growth frees the team from the manual work behind 'post-raise growth targets outpace what manual operations can deliver'.",
  "evidence_refs": [
    "https://techcrunch.com/example-funding"
  ]
}

Subject: an AI workflow platform for Rescale
Hi John,

I noticed Rescale's recent funding: Rescale Recent Funding Round raises Series B.

We build an AI workflow platform for teams where post-raise growth targets outpace what manual operations can deliver; customers cut that manual work by a measurable margin within a quarter.

Worth a 20-minute conversation next week?


## 4. Puntuación del juez (rúbrica en `src/agents/quality_scorer.py`)

In [5]:
from src.agents.quality_scorer import RUBRIC
print(result.scores.model_dump_json(indent=2))
print()
print(RUBRIC[:600], '...')

{
  "personalization": 5,
  "accuracy": 4,
  "cta_clarity": 4,
  "overall": 4.33,
  "rationale": "heuristic: facts=4 claims=1 unsupported=0 cta_format=True cta_time=True",
  "judge": "heuristic"
}

Score the outreach email on three criteria, each an integer from 1 to 5.

1. PERSONALIZATION - does the email prove it was written for THIS company?
   1 = fully generic; could be sent to anyone ("I came across your company").
   2 = mentions the company name only.
   3 = references the industry or a vague fact ("you are growing fast").
   4 = references one specific, verifiable fact from the profile (a funding
       round, a named product, a named executive, an open role).
   5 = weaves two or more specific facts into a coherent reason to reach out now.
   Example 5: "Congrats on the $40M Se ...


## 5. Baselines y ablation sobre una muestra del eval set

In [6]:
from eval.ground_truth import load_ground_truth
from eval.run import run_eval
companies = load_ground_truth(ROOT / 'data/eval/yc_ground_truth_200.json')[:5]
report = asyncio.run(run_eval(companies=companies, components=components, settings=s))
print('mode:', report['mode'])
for sysres in report['systems']:
    acc = sysres['accuracy']
    print(f"{sysres['system']:22s} P={acc['precision']:.2f} R={acc['recall']:.2f} "
          f"tool_calls={sysres['tool_calls']['mean']:.1f} "
          f"judge_pers={sysres['judge_personalization']['mean']:.2f} "
          f"({sysres['judge']})")

mode: fallback determinista, sin LLM
template_only          P=0.00 R=0.00 tool_calls=0.0 judge_pers=2.00 (heuristic)
single_search_email    P=0.00 R=0.00 tool_calls=1.0 judge_pers=2.00 (heuristic)
agent_reflection_0     P=0.20 R=0.05 tool_calls=7.0 judge_pers=5.00 (heuristic)
agent_reflection_1     P=0.20 R=0.05 tool_calls=7.0 judge_pers=5.00 (heuristic)
agent_reflection_2     P=0.20 R=0.05 tool_calls=7.0 judge_pers=5.00 (heuristic)
agent_reflection_3     P=0.20 R=0.05 tool_calls=7.0 judge_pers=5.00 (heuristic)


## 6. Persistencia

El repositorio en memoria (o PostgreSQL con `DATABASE_URL`) guarda cada resultado por nombre de empresa; el API lo usa como caché.

In [7]:
from src.storage import build_repository
repo = build_repository(s)
asyncio.run(repo.save(result))
cached = asyncio.run(repo.get('rescale'))
print(repo.name, '->', cached.company_name, cached.trace_id)

memory -> Rescale 3f76df3cae7e49af9d53f2f68f53c97a


## Nota

Con stubs, los perfiles contienen datos enlatados (por ejemplo una ronda "Series B" genérica) y no describen la empresa real: ilustran el formato y la mecánica del pipeline. La tabla con Claude + Tavily se genera con `python -m eval.run` y queda en `eval/RESULTS.md`.